# Attention-Augmented CNN

Trains `model.attention_cnn.AttentionAugmentedCNN` for guided
4 km → 1 km MODIS LST downscaling. Adds channel + spatial attention
inside every residual block — should beat the plain CNN baseline if
attention helps the network attend to terrain-driven LST gradients.


## 0 — Setup

In [ ]:
# Colab: install deps. Local: skip if already installed.
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rasterio geopandas h5py huggingface_hub scikit-learn


In [ ]:
# Pull the model + dataloader code. On Colab we clone; locally we assume cwd is the repo.
import os, sys
if IN_COLAB:
    if not os.path.isdir('downscaling'):
        !git clone -q https://github.com/fresleven/downscaling.git
    %cd downscaling
    sys.path.insert(0, '.')
else:
    # Add repo root to path (notebook lives in notebooks/colabs/)
    sys.path.insert(0, os.path.abspath('../..'))


In [ ]:
from huggingface_hub import snapshot_download
DATA_ROOT = 'data'
snapshot_download(
    repo_id='akhot2/downscaling',
    repo_type='dataset',
    local_dir=DATA_ROOT,
)


## 1 — Datasets

In [ ]:
from model.dataset import DownscalingDataset, get_dataloaders

train_ds = DownscalingDataset(root=DATA_ROOT, split='train', download=False)
val_ds   = DownscalingDataset(root=DATA_ROOT, split='val',   download=False)
test_ds  = DownscalingDataset(root=DATA_ROOT, split='test',  download=False)

print(f'Scenes — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}')
print(f'HR shape: {train_ds.hr_shape}, LR shape: {train_ds.lr_shape}')
print(f'LULC classes ({len(train_ds.lulc_classes)}): {train_ds.lulc_classes.tolist()}')


## 2 — Training utilities

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=0)


def make_hr_cov(batch):
    return torch.cat([batch['ndvi'], batch['dem'], batch['lulc_onehot']], dim=1)


def masked_mse(pred, target, mask):
    diff2 = (pred - target) ** 2 * mask
    return diff2.sum() / mask.sum().clamp(min=1)


@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    rmses, maes = [], []
    for b in loader:
        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        pred = model(b['lr_lst'], make_hr_cov(b))
        for i in range(pred.shape[0]):
            m = b['valid_mask'][i, 0]
            if not m.any():
                continue
            d = pred[i, 0][m] - b['hr_lst'][i, 0][m]
            rmses.append(torch.sqrt((d ** 2).mean()).item())
            maes.append(d.abs().mean().item())
    return {'RMSE': float(np.mean(rmses)), 'MAE': float(np.mean(maes))}


def train_model(model, n_epochs=60, lr=1e-3, weight_decay=1e-5):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history, best = [], {'val_rmse': float('inf'), 'state': None, 'epoch': -1}
    for epoch in range(n_epochs):
        model.train()
        losses = []
        for b in train_loader:
            b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
            pred = model(b['lr_lst'], make_hr_cov(b))
            loss = masked_mse(pred, b['hr_lst'], b['valid_mask'])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        sched.step()
        v = eval_loader(model, val_loader)
        history.append({'epoch': epoch, 'train_mse': float(np.mean(losses)), **v})
        if v['RMSE'] < best['val_rmse']:
            best = {'val_rmse': v['RMSE'],
                    'state': {k: t.detach().cpu().clone() for k, t in model.state_dict().items()},
                    'epoch': epoch}
        if epoch % 5 == 0 or epoch == n_epochs - 1:
            print(f"epoch {epoch:3d}  train_mse={np.mean(losses):.3f}  "
                  f"val RMSE={v['RMSE']:.3f}  MAE={v['MAE']:.3f}  (best @ {best['epoch']})")
    model.load_state_dict(best['state'])
    return model, history, best


## 3 — Build & train

In [ ]:
from model.attention_cnn import AttentionAugmentedCNN

n_classes = train_ds.lulc_classes.shape[0]
cov_channels = 1 + 1 + n_classes  # NDVI + DEM + LULC one-hot

model = AttentionAugmentedCNN(cov_channels=cov_channels, base_channels=64,
                              n_lr_blocks=4, n_hr_blocks=6)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

model, history, best = train_model(model, n_epochs=60, lr=1e-3)
print(f"\nBest val RMSE = {best['val_rmse']:.3f}°C @ epoch {best['epoch']}")


## 4 — Loss curves

In [ ]:
import matplotlib.pyplot as plt
ep = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [h['train_mse'] for h in history], label='train MSE'); ax[0].grid(); ax[0].legend()
ax[1].plot(ep, [h['RMSE'] for h in history], label='val RMSE'); ax[1].grid(); ax[1].legend()
ax[1].set_xlabel('epoch'); plt.show()


## 5 — Test evaluation & sample predictions

In [ ]:
test_metrics = eval_loader(model, test_loader)
print('Test:', test_metrics)

import matplotlib.pyplot as plt

# Pull one example
batch = next(iter(test_loader))
b_dev = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
with torch.no_grad():
    pred = model(b_dev['lr_lst'], make_hr_cov(b_dev)).cpu()

n_show = min(4, pred.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3*n_show, 7))
if n_show == 1: axes = axes[:, None]
for i in range(n_show):
    hr = batch['hr_lst'][i, 0].numpy()
    lr = batch['lr_lst'][i, 0].numpy()
    pr = pred[i, 0].numpy()
    vmin, vmax = float(hr.min()), float(hr.max())
    for r, (img, lbl) in enumerate([(lr, 'LR (4km)'), (pr, 'Pred (1km)'), (hr, 'HR (1km)')]):
        ax = axes[r, i]; ax.imshow(img, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0: ax.set_ylabel(lbl)
    axes[0, i].set_title(batch['date'][i], fontsize=9)
plt.tight_layout(); plt.show()


## 6 — Save weights

In [ ]:
torch.save(model.state_dict(), 'attention_cnn.pt')
print('Saved attention_cnn.pt')
